# Polly HF SFT Training (Kaggle)

Runs `scripts/train/hf_train_polly.py` from the SCBE-AETHERMOORE repo against `issdandavis/polly-training-data` and pushes the LoRA adapter to `issdandavis/polly-1`.

**Setup:** In Kaggle: *Add-ons → Secrets* → add a secret named `HF_TOKEN` with a Hugging Face token that has *write* scope. Attach a GPU accelerator (T4 x2 or P100 both work).

In [ ]:
# 1. Clone the training branch and install deps
import os, subprocess, sys
REPO_DIR = '/kaggle/working/scbe'
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', 'neurogolf/ant-colony-solvers',
                           'https://github.com/issdandavis/SCBE-AETHERMOORE.git', REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at', REPO_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', '-U',
    'transformers>=4.45', 'trl>=0.12', 'peft>=0.13', 'accelerate>=1.0',
    'bitsandbytes>=0.44', 'datasets>=3.0', 'huggingface_hub>=0.25'])

In [ ]:
# 2. Pull HF_TOKEN from Kaggle Secrets into the environment
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'HF_TOKEN missing'
print('HF_TOKEN loaded from Kaggle Secrets')

In [ ]:
# 3. GPU + dataset sanity check
import torch, os
print('cuda:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu{i}: {p.name} | {p.total_memory/1e9:.1f} GB | bf16={torch.cuda.is_bf16_supported()}')
from datasets import load_dataset
ds = load_dataset('issdandavis/polly-training-data', split='train', token=os.environ['HF_TOKEN'])
print('rows:', len(ds), '| cols:', ds.column_names)

In [ ]:
# 4. Train and push to issdandavis/polly-1
# Profile flags by GPU:
#   T4 single (15 GB):  --batch-size 2 --grad-accum 8
#   T4 x2 / P100 (16 GB):  --batch-size 2 --grad-accum 8
#   L4 (22 GB):  --batch-size 4 --grad-accum 4
#   A100 (40+):  --batch-size 8 --grad-accum 2 --no-quant
import subprocess, sys, os
os.chdir('/kaggle/working/scbe')
cmd = [sys.executable, '-u', 'scripts/train/hf_train_polly.py',
       '--base-model', 'Qwen/Qwen2.5-0.5B',
       '--output-repo', 'issdandavis/polly-1',
       '--epochs', '3',
       '--batch-size', '2',
       '--grad-accum', '8',
       '--lr', '2e-4',
       '--max-seq', '1024',
       '--max-grad-norm', '1.0',
       '--push']
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line); sys.stdout.flush()
p.wait()
print('RC=', p.returncode)
assert p.returncode == 0, 'training failed'

In [ ]:
# 5. Smoke check the pushed adapter
import os
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
info = api.repo_info('issdandavis/polly-1', repo_type='model')
print('repo:', info.id, '| sha:', info.sha[:8], '| files:', len(info.siblings))
for s in info.siblings[:20]:
    print(' -', s.rfilename)